0. Importing libraries  and loading datasets into dataframes

In [130]:
import pandas as pd
import numpy as np


1. Import the order dataset and print the first 5 rows

In [1]:
dfo = pd.read_excel("./orders.xlsx")
dfo.head()

NameError: name 'pd' is not defined

1. Import the customer dataset and print the first 5 rows

In [ ]:
dfc = pd.read_excel("./data/customers.xlsx")
dfc.head()

,CustomerID,Name,Location,Province,Type,Since
0,C0001,Amina Yusuf,Grande Prairie,Alberta,Standard,2020-12-29
1,C0002,Mateo Tremblay,Leduc,Alberta,VIP,2021-07-03
2,C0003,Priya Nair,Grande Prairie,Alberta,Standard,2019-07-01
3,C0004,Liam Chen,Sherwood Park,AB,Standard,2024-11-14
4,C0005,Sofia Kowalski,Calgary,Alberta,Standard,2023-10-27


Diagnosing the Data Quality

Though this is not for marks I find it important to identify the shape and datatypes of my datasets

Orders shape, datatypes and null value count

In [133]:
print(dfo.shape)
print(dfo.dtypes)
print(dfo.isna().sum())

(100, 7)
OrderID           str
CustomerID        str
Date           object
Item              str
Category          str
Qty           float64
Cost          float64
dtype: object
OrderID       0
CustomerID    1
Date          0
Item          0
Category      0
Qty           1
Cost          0
dtype: int64


Customers shape, datatypes and null value count

In [134]:
print(dfc.shape)
print(dfc.dtypes)
print(dfc.isna().sum())

(40, 6)
CustomerID               str
Name                     str
Location                 str
Province                 str
Type                     str
Since         datetime64[us]
dtype: object
CustomerID    0
Name          0
Location      0
Province      0
Type          1
Since         0
dtype: int64


Orders Missing-Value Report
The next cell calculates both how many values are missing and the percentage of values that are missing in each column

In [135]:
missing_summary_orders = pd.DataFrame({
    "MissingCount": dfo.isna().sum(), 
    "MissingPercent": dfo.isna().sum()
})
missing_summary_orders.sort_values("MissingPercent", ascending=False)

,MissingCount,MissingPercent
CustomerID,1,1
Qty,1,1
OrderID,0,0
Date,0,0
Item,0,0
Category,0,0
Cost,0,0


Customers Missing-Value Report
The next cell calculates both how many values are missing and the percentage of values that are missing in each column

In [136]:
missing_summary_customers = pd.DataFrame({
    "MissingCount": dfc.isna().sum(), 
    "MissingPercent": dfc.isna().sum()
})
missing_summary_customers.sort_values("MissingPercent", ascending=False)

,MissingCount,MissingPercent
Type,1,1
CustomerID,0,0
Name,0,0
Location,0,0
Province,0,0
Since,0,0


2. The following cell chains the .duplicated() and sum() methods from Pandas to first find all duplicate rows in the Orders dataset. Then count the number of rows that are duplicates. Finally printing out the quantity of duplicate rows. The .duplicated() method identifies the duplicate rows

In [137]:
duplicate_count = dfo.duplicated().sum()
print("Duplicate Order rows:", duplicate_count)

Duplicate Order rows: 1


2. The following cell chains the .duplicated() and sum() methods from Pandas to first find all duplicate rows in the Customers dataset. Then count the number of rows that are duplicates. Finally printing out the quantity of duplicate rows. 

Spoiler: There are no duplicate customer rows.

In [138]:
duplicate_count = dfc.duplicated().sum()
print("Duplicate Customer rows:", duplicate_count)

Duplicate Customer rows: 0


2. Because there's only one duplicate row across both datasets we simply call .drop_duplicates() on the dataframe that has duplicates


In [139]:
dfo.drop_duplicates()

,OrderID,CustomerID,Date,Item,Category,Qty,Cost
0,O1001,C0034,2026-08-01 00:00:00,Rechargeable Light Set,Lights,2.0,79.99
1,O1001,C0034,2026-08-01 00:00:00,Trail Helmet,Helmets,1.0,89.99
2,O1001,C0034,2026-08-01 00:00:00,Bike Multi-Tool,Tools,1.0,34.99
3,O1002,C0037,2026-05-02 00:00:00,Commuter Helmet,Helmets,1.0,69.99
4,O1002,C0037,2026-05-02 00:00:00,Rear Safety Light,Lights,1.0,24.99
...,...,...,...,...,...,...,...
94,O1033,C0022,2026-05-02 00:00:00,Saddle Bag,Accessories,1.0,44.99
95,O1033,C0022,2026-05-02 00:00:00,Hybrid Fitness Bike,Bicycles,1.0,1099.99
96,O1034,C0006,2026-01-08 00:00:00,Commuter Helmet,Helmets,1.0,69.99
97,O1034,C0006,2026-01-08 00:00:00,Rechargeable Light Set,Lights,1.0,79.99


3.1 To fix the missing customerID we can assign the column 'CustomerID' to be the value of the other CustomerID's within a matching OrderID. This is done by chaining the Pandas imputation methods .ffill().bfill() on the 'CustomerID' column of the dataframe while grouping the dataframe by 'OrderID'

In [140]:
dfo['CustomerID'] = dfo.groupby('OrderID')['CustomerID'].ffill().bfill()

3.2 To fix the missing Quantity from order O1003 you can infer based on the unit price of the U-Lock in O1011 that the quantity is 1 as the Cost is the same. I manually assign the Qty of 1 by index using the OrderId and Item description

In [141]:
dfo.loc[(dfo['OrderID'] == 'O1003') & (dfo['Item'] == 'U-Lock'), 'Qty'] = 1

3.3 To fix the missing Customer Type I figure the mode of all other types which is standard then I assign the missing Type entries with the modal type by calling the .fillna().

In [142]:
mode_type = dfc['Type'].mode()[0]
dfc['Type'] = dfc['Type'].fillna(mode_type)

4.1 The consistent style and name convention will be snake_case.
OrderId is a clear identifier but doesn't fit the style > order_id
CustomerId same as above > customer_id
Date is stored as an object rather than a time stamp and is ambiguous plus conflicts with both Python/SQL reserved words > order_date
Item/Categor clear but don't fit chosen style > item_name and category_name
Qty abbreviated name should be expanded for clarity > quantity
Cost slightly ambiguous and could be confused with total_cost so renamed to > unit_cost

In [143]:
dfo = dfo.rename(columns={
    'OrderID': 'order_id',
    'CustomerID': 'customer_id',
    'Date': 'order_date',
    'Item': 'item_name',
    'Category': 'category_name',
    'Qty': 'quantity',
    'Cost': 'unit_cost'
})
dfo.dtypes

order_id             str
customer_id          str
order_date        object
item_name            str
category_name        str
quantity         float64
unit_cost        float64
dtype: object

4.2 
CustomerID same as above > customer_id
Name should be distinguished from product names > customer_name
Location should be disambiguated > city
Province should be brought in line with the rest > province
Type is very ambiguous and should relate to the various tiers > customer_tier
Since is has temporal connotations but isn't grounded > join_date

In [144]:
dfc = dfc.rename(columns={
    'CustomerID': 'customer_id',
    'Name': 'customer_name',
    'Location': 'city',
    'Province': 'province',
    'Type': 'customer_tier',
    'Since': 'join_date'
})
dfc.dtypes

customer_id                 str
customer_name               str
city                        str
province                    str
customer_tier               str
join_date        datetime64[us]
dtype: object

5. The column result can be created by giving it a name and it's value can be assigned by row with =. In this case we're taking the product of two other rows(quantity and unit_cost) and assigning it to the column line_total that will be newly created

In [145]:
dfo['line_total'] = dfo['quantity'] * dfo['unit_cost']
dfo.head

<bound method NDFrame.head of    order_id customer_id           order_date               item_name  \
0     O1001       C0034  2026-08-01 00:00:00  Rechargeable Light Set   
1     O1001       C0034  2026-08-01 00:00:00            Trail Helmet   
2     O1001       C0034  2026-08-01 00:00:00         Bike Multi-Tool   
3     O1002       C0037  2026-05-02 00:00:00         Commuter Helmet   
4     O1002       C0037  2026-05-02 00:00:00       Rear Safety Light   
..      ...         ...                  ...                     ...   
95    O1033       C0022  2026-05-02 00:00:00     Hybrid Fitness Bike   
96    O1034       C0006  2026-01-08 00:00:00         Commuter Helmet   
97    O1034       C0006  2026-01-08 00:00:00  Rechargeable Light Set   
98    O1017       C0008  2026-09-11 00:00:00            Trail Helmet   
99    O1004       C0005  2026-01-30 00:00:00        Road Tire 700x28   

   category_name  quantity  unit_cost  line_total  
0         Lights       2.0      79.99      159.98  
1

6.a Because the sales orders is essentially bi-modal in it's distribution but has the carbon road bike outlier we need to choose min-max scaling the data is heavily right-skewed by the road bike outlier meaning that the z-score standardization would be representing a mean that doesn't actually represent the data. In this case the groupings are the components and complete bicycles with the outlier of the carbon road bike pulling the mean up higher than is representative.

In [146]:
dfo['unit_cost_minmax'] = (dfo['unit_cost'] - dfo['unit_cost'].min()) / (dfo['unit_cost'].max() - dfo['unit_cost'].min())
dfo.head

<bound method NDFrame.head of    order_id customer_id           order_date               item_name  \
0     O1001       C0034  2026-08-01 00:00:00  Rechargeable Light Set   
1     O1001       C0034  2026-08-01 00:00:00            Trail Helmet   
2     O1001       C0034  2026-08-01 00:00:00         Bike Multi-Tool   
3     O1002       C0037  2026-05-02 00:00:00         Commuter Helmet   
4     O1002       C0037  2026-05-02 00:00:00       Rear Safety Light   
..      ...         ...                  ...                     ...   
95    O1033       C0022  2026-05-02 00:00:00     Hybrid Fitness Bike   
96    O1034       C0006  2026-01-08 00:00:00         Commuter Helmet   
97    O1034       C0006  2026-01-08 00:00:00  Rechargeable Light Set   
98    O1017       C0008  2026-09-11 00:00:00            Trail Helmet   
99    O1004       C0005  2026-01-30 00:00:00        Road Tire 700x28   

   category_name  quantity  unit_cost  line_total  unit_cost_minmax  
0         Lights       2.0      79.

7. Using the Python .dt.strftime() method we can format a string into an appropriate date format by calling it on the column

In [147]:
dfo['date_normalized'] = pd.to_datetime(dfo['order_date'])
dfo.head

<bound method NDFrame.head of    order_id customer_id           order_date               item_name  \
0     O1001       C0034  2026-08-01 00:00:00  Rechargeable Light Set   
1     O1001       C0034  2026-08-01 00:00:00            Trail Helmet   
2     O1001       C0034  2026-08-01 00:00:00         Bike Multi-Tool   
3     O1002       C0037  2026-05-02 00:00:00         Commuter Helmet   
4     O1002       C0037  2026-05-02 00:00:00       Rear Safety Light   
..      ...         ...                  ...                     ...   
95    O1033       C0022  2026-05-02 00:00:00     Hybrid Fitness Bike   
96    O1034       C0006  2026-01-08 00:00:00         Commuter Helmet   
97    O1034       C0006  2026-01-08 00:00:00  Rechargeable Light Set   
98    O1017       C0008  2026-09-11 00:00:00            Trail Helmet   
99    O1004       C0005  2026-01-30 00:00:00        Road Tire 700x28   

   category_name  quantity  unit_cost  line_total  unit_cost_minmax  \
0         Lights       2.0      79

8. Using the pandas.merge() function we call it on the dataframes specifying the column on which to join with an inner join.

In [148]:
dfm = pd.merge(dfo, dfc, on='customer_id', how='inner')

8.a Showing the merged DataFrame by calling it

In [149]:
dfm

,order_id,customer_id,order_date,item_name,category_name,quantity,unit_cost,line_total,unit_cost_minmax,date_normalized,customer_name,city,province,customer_tier,join_date
0,O1001,C0034,2026-08-01 00:00:00,Rechargeable Light Set,Lights,2.0,79.99,159.98,0.016317,2026-08-01,Arjun Rao,Medicine Hat,Alberta,Std,2018-01-08
1,O1001,C0034,2026-08-01 00:00:00,Trail Helmet,Helmets,1.0,89.99,89.99,0.018648,2026-08-01,Arjun Rao,Medicine Hat,Alberta,Std,2018-01-08
2,O1001,C0034,2026-08-01 00:00:00,Bike Multi-Tool,Tools,1.0,34.99,34.99,0.005828,2026-08-01,Arjun Rao,Medicine Hat,Alberta,Std,2018-01-08
3,O1002,C0037,2026-05-02 00:00:00,Commuter Helmet,Helmets,1.0,69.99,69.99,0.013986,2026-05-02,Yuna Sato,Spruce Grove,Alberta,Standard,2018-06-04
4,O1002,C0037,2026-05-02 00:00:00,Rear Safety Light,Lights,1.0,24.99,24.99,0.003497,2026-05-02,Yuna Sato,Spruce Grove,Alberta,Standard,2018-06-04
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94,O1033,C0022,2026-05-02 00:00:00,Hybrid Fitness Bike,Bicycles,1.0,1099.99,1099.99,0.254079,2026-05-02,Theo Johnson,Lethbridge,Alberta,Premium,2020-04-01
95,O1034,C0006,2026-01-08 00:00:00,Commuter Helmet,Helmets,1.0,69.99,69.99,0.013986,2026-01-08,Noah Cardinal,Edmonton,Alberta,Premium,2022-08-15
96,O1034,C0006,2026-01-08 00:00:00,Rechargeable Light Set,Lights,1.0,79.99,79.99,0.016317,2026-01-08,Noah Cardinal,Edmonton,Alberta,Premium,2022-08-15
97,O1017,C0008,2026-09-11 00:00:00,Trail Helmet,safety gear,1.0,89.99,89.99,0.018648,2026-09-11,Ethan Nguyen,Leduc,Alberta,standard,2019-02-01


9. Calculate the total cost per order

In [150]:
order_totals = dfm.groupby('order_id')['line_total'].sum()
print(order_totals)

order_id
O1001     284.96
O1002     454.95
O1003     149.97
O1004    1149.94
O1005     229.97
O1006     254.96
O1007     169.96
O1008     214.96
O1009     354.95
O1010     829.91
O1011    1179.97
O1012   -2504.99
O1013    1229.97
O1014    1819.95
O1015     384.92
O1016    2954.96
O1017    2394.95
O1018    1184.97
O1019     169.97
O1020     304.86
O1021    4224.95
O1022    1969.96
O1023     149.97
O1024    2304.96
O1025     164.98
O1026     259.95
O1027     134.96
O1028     184.97
O1029    4384.95
O1030    2404.96
O1031     219.96
O1032     229.97
O1033    1144.98
O1034     149.98
Name: line_total, dtype: float64


9. Calculate the mean and median cost per order

In [151]:
mean_order_cost = order_totals.mean()
median_order_cost = order_totals.median()
print(f"Mean Total Cost per Order: ${mean_order_cost:.2f}")
print(f"Median Total Cost per Order: ${median_order_cost:.2f}")

Mean Total Cost per Order: $915.99
Median Total Cost per Order: $329.91


10. Identify one outlier order

In [152]:
df_order_totals = dfm.groupby('order_id')['line_total'].sum().reset_index() # Calling .reset_index() to get a full dataframe instead of a pandas series

Q1 = df_order_totals['line_total'].quantile(0.25)
Q3 = df_order_totals['line_total'].quantile(0.75)
IQR = Q3 - Q1

lower_fence = Q1 - 1.5 * IQR
upper_fence = Q3 + 1.5 * IQR

iqr_outliers = df_order_totals[(df_order_totals['line_total'] < lower_fence) | 
                            (df_order_totals['line_total'] > upper_fence)]
print("IQR Outliers:")
print(iqr_outliers)

IQR Outliers:
   order_id  line_total
11    O1012    -2504.99
15    O1016     2954.96
20    O1021     4224.95
28    O1029     4384.95


I found 4 outliers. 
O1012 is a return of three bikes that is WAY smaller than all the other orders as it represents the return of three mid-range bikes. 
O1016 is an outlier on the high side because it's the purchase of three bikes.
O1021 is another outlier on the high side because it's the purchase of 4 bikes.
O1029 is the final outlier on the high side because it's a high-end bike which is uncommon for this shop.